# Running local LLMs

### Install Dependencies

In [ ]:
!apt update && apt-get install zstd pciutils lshw
!curl -fsSL https://ollama.com/install.sh | sh

!pip install ollama

# !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
# !chmod +x /usr/local/bin/cloudflared

### Run ollama in the background

In [ ]:
import os
os.system(
    "OLLAMA_HOST=0.0.0.0:11434 OLLAMA_KEEP_ALIVE=30m "
    "OLLAMA_NUM_PARALLEL=1 OLLAMA_MAX_LOADED_MODELS=1 "
    "OLLAMA_CONTEXT_LENGTH=65000 "
    "OLLAMA_NUM_PARALLEL=1 "
    "OLLAMA_FLASH_ATTENTION=1 "
    "OLLAMA_KV_CACHE_TYPE=q8_0 "
    # "OLLAMA_NUM_THREADS 4 "
    "nohup ollama serve > /dev/null 2>&1 &"
)

### Check Ollama server

In [ ]:
!ollama list

### Pull some models

(can take 1 to 3 minutes per model)

(run these on the terminal so there's a progress bar)

In [ ]:
!ollama pull gemma4:e4b

### Check if ollama server is still running

In [ ]:
!ollama list

### Inference Setup

In [ ]:
import json

from ollama import Client
from pydantic import BaseModel
from typing import Literal

OLLAMA_URL = "http://127.0.0.1:11434"

model = Client(host=OLLAMA_URL)

### Text Query

In [ ]:
response = model.chat(
  model="gemma4:e4b",
  options={"temperature": 0},
  messages=[{
    "role": "user",
    "content": "Why are eggs yellow?",
  }],
  think=False
)

# print(response)
print(response["message"]["content"])

### Using structured input and output

In [ ]:
from enum import Enum
from pydantic import BaseModel
from typing import Literal

In [ ]:
class GenreEnum(str, Enum):
  COMEDY = "comedy"
  ROMCOM = "romantic comedy"
  HORROR = "horror"

class OneGenre(BaseModel):
  genre: GenreEnum

In [ ]:
class Genres(BaseModel):
  genres: list[Literal["comedy", "romantic comedy", "horror"]]

descriptions = [
  "The film stars Sissy Spacek as Carrie White, a shy teenage girl who is constantly mocked and bullied at her school. She later develops the power of telekinesis and uses it to wreak vengeance on her tormentors.",
  "After initially hating one another, Harry and Sally eventually decide to be friends. Set over a sprawling time period, Harry and Sally’s friendship never fades, even when their lives become incredibly complicated.",
  "When the tiny nation of Freedonia goes bankrupt, its wealthy benefactor, Mrs. Teasdale, insists that the wacky Rufus T. Firefly become the country's president and plenty of mayhem ensues.",
  "A chance meeting between paleontologist David and the spontaneous Susan. Believing David is a zoologist, Susan convinces him to accompany her to Connecticut to take care of her new leopard, Baby."
]

In [ ]:
Genres.model_json_schema()

In [ ]:
response = model.chat(
  model="gemma4:e4b",
  format=Genres.model_json_schema(),
  options={"temperature": 0},
  messages=[
    {
      "role": "system",
      "content": "You are a helpful assistant that categorizes movies based on their plot summary."
    },
    {
      "role": "user",
      "content": str(descriptions),
    }
  ],
  think=False
)

genres = json.loads(response["message"]["content"])["genres"]
print(genres)

### Images

In [ ]:
from PIL import Image as PImage
from utils.utils import image_to_b64

In [ ]:
img = PImage.open("../imgs/gudetama.jpg")
display(img)

img64 = image_to_b64(img)["data"]
print(img64[:64])

In [ ]:
response = model.chat(
  model="gemma4:e4b",
  options={"temperature": 0},
  messages=[{
      "role": "user",
      "content": "Describe this image using a list of nouns and adjectives in plain text, separated by commas. No json, no markdow and no special or escape characters.",
      "images": [img64],
  }],
  think=False
)

response_content = response["message"]["content"]
print(response_content)

## Exercise

Repeat the exercises from `LAB07`, where we ran data from `data/astrolabe.json` or `data/misc.json` through an LLM for classification, extraction, or description.

In [ ]:
# TODO: instantiate chat model
#       open json
#       iterate and send questions to chat
#       add answers to json